# Анализ CRM по сегментам бизнеса

Источник: `crm_last_version.csv` + `ref_book.csv` (справочник ФП/СФП)

| X_AREA_RESP | Сегмент |
|-------------|---------|
| ДМСБ (микро) | МкБ |
| ДМСБ (малый) | МСБ |
| ДМСБ (средний) | МСБ |
| ДКБ | ККБ |

**Период:** 2023–2025 | **Дедупликация:** INN + NUMBER_FP_SFP + IDENTIFICATION_DATE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ", "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ", "ДКБ": "ККБ",
}

# Маппинг X_AREA_RESP → ЗО из справочника (наименования отличаются)
ZO_MAP = {
    "ДМСБ (микро)": "ДМ (микро)", "ДМСБ (малый)": "ДМСБ (малый)",
    "ДМСБ (средний)": "ДМСБ (средний)", "ДКБ": "ДКБ",
}

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

def normalize_inn(s):
    """Нормализация ИНН: убирает .0 и ведущие нули, дополняет до 10/12 знаков."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

## 1. Загрузка и подготовка данных

In [ ]:
# Загрузка (все колонки как строки, чтобы не терять ведущие нули в ИНН/КПП)
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
print(f"Загружено: {len(df):,} строк")

if "VAL" in df.columns:
    before_src = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам (VAL): {len(df):,} (удалено {before_src - len(df):,})")

df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

# Фильтр по периоду
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра {DATE_FROM:%d.%m.%Y} – {DATE_TO:%d.%m.%Y}: {len(df):,} строк")

# В реальной выгрузке TYPE_FP = "FP"/"SFP" → приводим к кириллице
df["TYPE_FP"] = df["TYPE_FP"].replace({"FP": "ФП", "SFP": "СФП"})

# Маппинг сегментов
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print(f"\nНемаппированные X_AREA_RESP (исключены):\n{unmapped.to_string()}")

df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

# Сохраняем состояние до дедупликации для диагностики ROW_ID (следующая ячейка)
df_before_dedup = df.copy()

# Дедупликация (одинаковый ИНН + фактор + дата = дубль; разные даты = разные события)
before = len(df)
df = df.drop_duplicates(subset=["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации: {len(df):,} строк (удалено {before - len(df):,})")

# Удаление строк без номера фактора / ИНН
before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

df["year_month"] = df["dt"].dt.to_period("M")

## 1.1 Диагностика: влияние ROW_ID на дедупликацию

Сравниваем два ключа дедупликации:
- **Ключ A** (текущий): `INN + NUMBER_FP_SFP + IDENTIFICATION_DATE`
- **Ключ B** (расширенный): `INN + NUMBER_FP_SFP + IDENTIFICATION_DATE + ROW_ID`

Если дельта > 0, значит существуют случаи, когда один и тот же ФП у одного ИНН в один день имеет **разные карточки** (разные `ROW_ID`).

In [ ]:
KEY_A = ["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]
KEY_B = ["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE", "ROW_ID"]

n_raw = len(df_before_dedup)
n_key_a = df_before_dedup.drop_duplicates(subset=KEY_A).shape[0]
n_key_b = df_before_dedup.drop_duplicates(subset=KEY_B).shape[0]

delta = n_key_b - n_key_a

print("=" * 70)
print("СРАВНЕНИЕ КЛЮЧЕЙ ДЕДУПЛИКАЦИИ")
print("=" * 70)
print(f"  Строк до дедупликации:                        {n_raw:>12,}")
print(f"  Ключ A (INN + FP + DATE):                     {n_key_a:>12,}")
print(f"  Ключ B (INN + FP + DATE + ROW_ID):            {n_key_b:>12,}")
print(f"  Дельта (записи с разным ROW_ID):               {delta:>12,}")
print(f"  Дельта в % от ключа A:                         {delta / n_key_a * 100:>11.2f}%")

# Находим группы, где одинаковые (inn, FP, date) имеют >1 ROW_ID
dup_groups = (
    df_before_dedup
    .groupby(KEY_A)["ROW_ID"]
    .nunique()
    .reset_index(name="n_row_ids")
)
multi_rid = dup_groups[dup_groups["n_row_ids"] > 1]

print(f"\n  Групп (INN + FP + DATE) с >1 ROW_ID:           {len(multi_rid):>12,}")
if len(multi_rid) > 0:
    print(f"  Макс. ROW_ID в одной группе:                   {multi_rid['n_row_ids'].max():>12,}")

# Примеры: до 10 групп с наибольшим числом разных ROW_ID
if len(multi_rid) > 0:
    examples = multi_rid.nlargest(10, "n_row_ids")
    print(f"\n{'='*70}")
    print(f"ПРИМЕРЫ ГРУПП С НЕСКОЛЬКИМИ ROW_ID (до 10)")
    print(f"{'='*70}")
    for _, row in examples.iterrows():
        inn, fp, dt, n = row["inn"], row["NUMBER_FP_SFP"], row["IDENTIFICATION_DATE"], row["n_row_ids"]
        print(f"\n  ИНН: {inn}  |  ФП: {fp}  |  Дата: {dt}  |  ROW_ID: {n} шт.")
        subset = df_before_dedup[
            (df_before_dedup["inn"] == inn) &
            (df_before_dedup["NUMBER_FP_SFP"] == fp) &
            (df_before_dedup["IDENTIFICATION_DATE"] == dt)
        ][["ROW_ID", "AGREEMENT_NUM", "TYPE_FP", "VAL"]].head(10)
        display(subset.reset_index(drop=True))
else:
    print("\n  Все группы (INN + FP + DATE) имеют ровно один ROW_ID — дельты нет.")

del df_before_dedup

## 2. Проверка: ИНН в нескольких сегментах

In [ ]:
seg_per_inn = df.groupby("inn")["segment"].nunique()
multi_seg = seg_per_inn[seg_per_inn > 1]

print("=" * 60)
print("ПРОВЕРКА: ИНН в нескольких сегментах")
print("=" * 60)
print(f"  Всего уникальных ИНН:              {len(seg_per_inn):,}")
print(f"  ИНН в одном сегменте:              {(seg_per_inn == 1).sum():,}")
print(f"  ИНН в нескольких сегментах:        {len(multi_seg):,}")

if len(multi_seg) > 0:
    print(f"\n  Детали (до 15 примеров):")
    print("  " + "-" * 56)
    for inn in multi_seg.index[:15]:
        s = df[df["inn"] == inn]
        print(f"  ИНН {inn}  ->  {len(s)} записей")
        print(f"    X_AREA_RESP: {', '.join(s['X_AREA_RESP'].unique())}")
        print(f"    Сегменты:    {', '.join(s['segment'].unique())}")
else:
    print("\n  Каждый ИНН принадлежит ровно одному сегменту.")

## 3. Уникальные ИНН по сегментам

In [ ]:
total_inn = df["inn"].nunique()
seg_order = ["МкБ", "МСБ", "ККБ"]
inn_by_seg = df.groupby("segment")["inn"].nunique().reindex(seg_order, fill_value=0)

print("=" * 50)
print("УНИКАЛЬНЫЕ ИНН")
print("=" * 50)
print(f"  Всего:  {total_inn:,}")
print("  " + "-" * 30)
for seg in seg_order:
    cnt = inn_by_seg.get(seg, 0)
    print(f"  {seg:<6s} {cnt:>8,}  ({cnt / total_inn * 100:.1f}%)")

tbl = pd.DataFrame({"Сегмент": seg_order, "Уникальных ИНН": [inn_by_seg.get(s, 0) for s in seg_order]})
tbl.loc[len(tbl)] = ["ИТОГО", total_inn]
display(tbl.style.hide(axis="index"))

## 4. Общая статистика по ФП/СФП

In [ ]:
total_factors = df["NUMBER_FP_SFP"].nunique()
fp_factors  = df[df["TYPE_FP"] == "ФП"]["NUMBER_FP_SFP"].nunique()
sfp_factors = df[df["TYPE_FP"] == "СФП"]["NUMBER_FP_SFP"].nunique()

fp_events  = (df["TYPE_FP"] == "ФП").sum()
sfp_events = (df["TYPE_FP"] == "СФП").sum()

print("=" * 60)
print("ОБЩАЯ СТАТИСТИКА ПО ФП/СФП")
print("=" * 60)
print(f"  Всего записей (событий):              {len(df):,}")
print(f"    из них ФП:                           {fp_events:,}")
print(f"    из них СФП:                          {sfp_events:,}")
print()
print(f"  Уникальных номеров факторов:           {total_factors}")
print(f"    встречаются как ФП:                  {fp_factors}")
print(f"    встречаются как СФП:                 {sfp_factors}")

In [ ]:
fp_set = set(df[df["TYPE_FP"] == "ФП"]["NUMBER_FP_SFP"].dropna().unique())
sfp_set = set(df[df["TYPE_FP"] == "СФП"]["NUMBER_FP_SFP"].dropna().unique())
both = sorted(fp_set & sfp_set)

print("=" * 70)
print("ПРОВЕРКА: факторы, встречающиеся и как ФП, и как СФП")
print("=" * 70)
print(f"  Только ФП:          {len(fp_set - sfp_set)}")
print(f"  Только СФП:         {len(sfp_set - fp_set)}")
print(f"  В обоих типах:      {len(both)}")
print(f"  Итого уникальных:   {len(fp_set | sfp_set)}")
print()
if both:
    _ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
    _ref.columns = _ref.columns.str.strip()
    _name_map = _ref.drop_duplicates(subset=["№"]).set_index("№")["Наименование"].to_dict()
    for num in both:
        fp_cnt = (df[(df["NUMBER_FP_SFP"]==num)&(df["TYPE_FP"]=="ФП")]).shape[0]
        sfp_cnt = (df[(df["NUMBER_FP_SFP"]==num)&(df["TYPE_FP"]=="СФП")]).shape[0]
        name = _name_map.get(num, "— не найдено в справочнике —")
        print(f"    {num:<10s}  ФП: {fp_cnt:>6,}  |  СФП: {sfp_cnt:>5,}")
        print(f"              {name}\n")

## 5. Количество ФП/СФП по сегментам и среднее на ИНН

In [ ]:
fp_stats = df.groupby(["segment", "TYPE_FP"]).size().unstack(fill_value=0).reindex(seg_order, fill_value=0)
for col in ["ФП", "СФП"]:
    if col not in fp_stats.columns:
        fp_stats[col] = 0
fp_stats = fp_stats[["ФП", "СФП"]]
fp_stats["Всего"] = fp_stats["ФП"] + fp_stats["СФП"]
fp_stats.loc["ИТОГО"] = fp_stats.sum()

print("=" * 60)
print("КОЛИЧЕСТВО ФП/СФП ПО СЕГМЕНТАМ")
print("=" * 60)
display(fp_stats)

# Среднее количество ФП/СФП на один ИНН
avg_rows = []
for seg in seg_order:
    seg_df = df[df["segment"] == seg]
    n_inn = seg_df["inn"].nunique()
    if n_inn == 0:
        continue
    n_fp, n_sfp, n_all = (seg_df["TYPE_FP"] == "ФП").sum(), (seg_df["TYPE_FP"] == "СФП").sum(), len(seg_df)
    avg_rows.append({
        "Сегмент": seg, "Уник. ИНН": n_inn, "ФП": n_fp, "СФП": n_sfp, "Всего": n_all,
        "Ср. ФП/ИНН": round(n_fp / n_inn, 2), "Ср. СФП/ИНН": round(n_sfp / n_inn, 2),
        "Ср. всего/ИНН": round(n_all / n_inn, 2),
    })

n = df["inn"].nunique()
avg_rows.append({
    "Сегмент": "ИТОГО", "Уник. ИНН": n, "ФП": (df["TYPE_FP"]=="ФП").sum(),
    "СФП": (df["TYPE_FP"]=="СФП").sum(), "Всего": len(df),
    "Ср. ФП/ИНН": round((df["TYPE_FP"]=="ФП").sum()/n, 2),
    "Ср. СФП/ИНН": round((df["TYPE_FP"]=="СФП").sum()/n, 2),
    "Ср. всего/ИНН": round(len(df)/n, 2),
})

print("\nСРЕДНЕЕ КОЛИЧЕСТВО ФП/СФП НА ОДИН ИНН")
display(pd.DataFrame(avg_rows).style.hide(axis="index"))

# График
fp_plot = fp_stats.drop("ИТОГО")
ax = fp_plot[["ФП", "СФП"]].plot(kind="bar", figsize=(8, 4), edgecolor="white")
ax.set_title("Количество ФП и СФП по сегментам", fontsize=13, fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("Количество")
ax.set_xticklabels(seg_order, rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=9)
plt.tight_layout(); plt.show()

## 6. Топ-20 самых частых ФП

In [ ]:
_total = (df["TYPE_FP"]=="ФП").sum()
top = df[df["TYPE_FP"]=="ФП"].groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).head(20).reset_index()
top.columns = ["№ ФП", "Кол-во"]
top["% от всех ФП"] = (top["Кол-во"] / _total * 100).round(2).astype(str) + "%"
top.index = range(1, len(top)+1)
print(f"Всего ФП: {_total:,}")
display(top)

## 7. Топ-20 самых частых СФП

In [ ]:
_total = (df["TYPE_FP"]=="СФП").sum()
top = df[df["TYPE_FP"]=="СФП"].groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).head(20).reset_index()
top.columns = ["№ СФП", "Кол-во"]
top["% от всех СФП"] = (top["Кол-во"] / _total * 100).round(2).astype(str) + "%"
top.index = range(1, len(top)+1)
print(f"Всего СФП: {_total:,}")
display(top)

## 8. Топ-20 ФП по сегментам

In [ ]:
for seg in seg_order:
    seg_fp = df[(df["segment"]==seg)&(df["TYPE_FP"]=="ФП")]
    seg_total = len(seg_fp)
    t = seg_fp.groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).head(20).reset_index()
    t.columns = ["№ ФП", "Кол-во"]
    t["% от ФП сегмента"] = (t["Кол-во"] / seg_total * 100).round(2).astype(str) + "%"
    t.index = range(1, len(t)+1)
    print(f"\n{'='*55}\n  {seg}  —  Топ-20 ФП  (всего: {seg_total:,})\n{'='*55}")
    display(t)

## 9. Топ-20 СФП по сегментам

In [ ]:
for seg in seg_order:
    seg_sfp = df[(df["segment"]==seg)&(df["TYPE_FP"]=="СФП")]
    seg_total = len(seg_sfp)
    t = seg_sfp.groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).head(20).reset_index()
    t.columns = ["№ СФП", "Кол-во"]
    t["% от СФП сегмента"] = (t["Кол-во"] / seg_total * 100).round(2).astype(str) + "%" if seg_total > 0 else "—"
    t.index = range(1, len(t)+1)
    print(f"\n{'='*55}\n  {seg}  —  Топ-20 СФП  (всего: {seg_total:,})\n{'='*55}")
    display(t)

## 10. Подключение справочника ФП/СФП

In [ ]:
df_ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
df_ref.columns = df_ref.columns.str.strip()

df["zo"] = df["X_AREA_RESP"].map(ZO_MAP)

# Джойн по (номер фактора + ЗО) — один номер в разных ЗО имеет разное наименование
df_merged = df.merge(
    df_ref[["№", "ЗО для ФП/СФП", "Наименование"]],
    left_on=["NUMBER_FP_SFP", "zo"], right_on=["№", "ЗО для ФП/СФП"], how="left",
)
no_match = df_merged["Наименование"].isna().sum()
print(f"Джойн: {len(df_merged):,} строк, без наименования: {no_match:,} ({no_match/len(df_merged)*100:.1f}%)")

def build_top(data, n=5):
    """Топ-N по частоте NUMBER_FP_SFP с подтягиванием наименований."""
    counts = data.groupby("NUMBER_FP_SFP").size().nlargest(n)
    rows = []
    for num, cnt in counts.items():
        names = data.loc[data["NUMBER_FP_SFP"]==num, "Наименование"].dropna().unique()
        rows.append({"№ ФП/СФП": num, "Наименование": " / ".join(sorted(names)) if len(names) else "", "Кол-во": cnt})
    r = pd.DataFrame(rows); r.index = range(1, len(r)+1)
    return r

## 11. Топ-5 ФП по сегментам с наименованиями

In [ ]:
for seg in seg_order:
    d = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="ФП")]
    if len(d) == 0: continue
    zo = ", ".join(sorted(d["zo"].dropna().unique()))
    print(f"{'='*80}\n  {seg}  (ЗО: {zo})  —  Топ-5 ФП\n{'='*80}")
    display(build_top(d, 5)); print()

## 12. Топ-5 СФП по сегментам с наименованиями

In [ ]:
for seg in seg_order:
    d = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="СФП")]
    if len(d) == 0: continue
    zo = ", ".join(sorted(d["zo"].dropna().unique()))
    print(f"{'='*80}\n  {seg}  (ЗО: {zo})  —  Топ-5 СФП\n{'='*80}")
    display(build_top(d, 5)); print()

## 13. Самые частые комбинации из 2 ФП по сегментам
Для каждого ИНН берём все уникальные ФП, генерируем пары, считаем частоту.

In [ ]:
from itertools import combinations

for seg in seg_order:
    seg_fp = df_merged[(df_merged["segment"]==seg) & (df_merged["TYPE_FP"]=="ФП")]
    inn_factors = seg_fp.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))

    pair_counts = {}
    for factors in inn_factors:
        if len(factors) >= 2:
            for pair in combinations(factors, 2):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    if not pair_counts:
        print(f"\n  {seg}: нет ИНН с 2+ различными ФП\n"); continue

    top_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])[:10]
    rows = []
    for (f1, f2), cnt in top_pairs:
        n1 = seg_fp.loc[seg_fp["NUMBER_FP_SFP"]==f1, "Наименование"].dropna().unique()
        n2 = seg_fp.loc[seg_fp["NUMBER_FP_SFP"]==f2, "Наименование"].dropna().unique()
        rows.append({"ФП 1": f1, "Наименование 1": " / ".join(sorted(n1)) if len(n1) else "",
                      "ФП 2": f2, "Наименование 2": " / ".join(sorted(n2)) if len(n2) else "",
                      "Кол-во ИНН": cnt})
    r = pd.DataFrame(rows); r.index = range(1, len(r)+1)
    zo = ", ".join(sorted(seg_fp["zo"].dropna().unique()))
    print(f"{'='*95}\n  {seg}  (ЗО: {zo})  —  Топ комбинаций из 2 ФП\n{'='*95}")
    display(r); print()

## 14. Самые частые комбинации из 2 СФП по сегментам

## 15. Динамика ФП по месяцам (по сегментам)

In [ ]:
fp_dyn = df[df["TYPE_FP"]=="ФП"].groupby(["year_month", "segment"]).size().unstack(fill_value=0).reindex(columns=seg_order, fill_value=0)

tbl = fp_dyn.copy()
tbl.index = tbl.index.astype(str)
tbl["Итого"] = tbl[seg_order].sum(axis=1)
tbl.index.name = "Месяц"
display(tbl)

fig, ax = plt.subplots(figsize=(14, 5))
for seg in seg_order:
    if seg in fp_dyn.columns:
        ax.plot(fp_dyn.index.astype(str), fp_dyn[seg], marker="o", markersize=4, label=seg)
ax.set_title("Динамика ФП по месяцам", fontsize=13, fontweight="bold")
ax.set_ylabel("Количество ФП"); ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

## 16. Динамика СФП по месяцам (по сегментам)

In [ ]:
sfp_dyn = df[df["TYPE_FP"]=="СФП"].groupby(["year_month", "segment"]).size().unstack(fill_value=0).reindex(columns=seg_order, fill_value=0)

tbl = sfp_dyn.copy()
tbl.index = tbl.index.astype(str)
tbl["Итого"] = tbl[seg_order].sum(axis=1)
tbl.index.name = "Месяц"
display(tbl)

fig, ax = plt.subplots(figsize=(14, 5))
for seg in seg_order:
    if seg in sfp_dyn.columns:
        ax.plot(sfp_dyn.index.astype(str), sfp_dyn[seg], marker="o", markersize=4, label=seg)
ax.set_title("Динамика СФП по месяцам", fontsize=13, fontweight="bold")
ax.set_ylabel("Количество СФП"); ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

## 17. Анализ сумм по договорам (APPROVED_SUM)

Для каждого уникального ИНН суммируем `APPROVED_SUM` по всем уникальным договорам (`AGREEMENT_NUM`).
Учитываются только рублёвые договоры (`CURCY_CD == "RUB"`).
Результат распределяется по 8 корзинам в разрезе по сегментам.

In [ ]:
# --- Фильтр: только RUB ---
df_rub = df[df["CURCY_CD"].astype(str).str.strip().str.upper() == "RUB"].copy()
print(f"Строк с CURCY_CD = RUB: {len(df_rub):,} из {len(df):,}")

# --- Приведение APPROVED_SUM к числу ---
df_rub["approved_sum"] = pd.to_numeric(
    df_rub["APPROVED_SUM"].astype(str).str.replace(",", ".").str.strip(),
    errors="coerce"
)
print(f"APPROVED_SUM заполнен: {df_rub['approved_sum'].notna().sum():,} ({df_rub['approved_sum'].notna().mean():.1%})")

# --- Дедупликация по AGREEMENT_NUM + INN ---
df_agr = (
    df_rub[df_rub["approved_sum"].notna()]
    .drop_duplicates(subset=["inn", "AGREEMENT_NUM"])
    [["inn", "segment", "AGREEMENT_NUM", "approved_sum"]]
    .copy()
)
print(f"Уникальных пар ИНН + договор: {len(df_agr):,}")

# --- Суммирование по ИНН ---
inn_sum = df_agr.groupby("inn").agg(
    total_sum=("approved_sum", "sum"),
    n_agreements=("AGREEMENT_NUM", "nunique"),
    segment=("segment", "first"),
).reset_index()
print(f"Уникальных ИНН с суммой: {len(inn_sum):,}")

# --- Корзины ---
BUCKET_LABELS = [
    "до 100 млн",
    "100–500 млн",
    "500 млн – 1 млрд",
    "1–2 млрд",
    "2–5 млрд",
    "5–10 млрд",
    "10–20 млрд",
    "свыше 20 млрд",
]
BUCKET_EDGES = [0, 1e8, 5e8, 1e9, 2e9, 5e9, 1e10, 2e10, float("inf")]

inn_sum["bucket"] = pd.cut(
    inn_sum["total_sum"], bins=BUCKET_EDGES, labels=BUCKET_LABELS,
    right=True, include_lowest=True,
)

# --- Кросс-таблица: корзина × сегмент (абсолютные) ---
cross_abs = pd.crosstab(
    inn_sum["bucket"], inn_sum["segment"],
    margins=True, margins_name="Итого"
).reindex(columns=seg_order + ["Итого"], fill_value=0)
cross_abs = cross_abs.reindex(BUCKET_LABELS + ["Итого"], fill_value=0)

print("\n" + "=" * 60)
print("РАСПРЕДЕЛЕНИЕ ИНН ПО КОРЗИНАМ APPROVED_SUM (абсолютные)")
print("=" * 60)
display(cross_abs)

# --- Кросс-таблица: корзина × сегмент (проценты внутри сегмента) ---
cross_pct = cross_abs.div(cross_abs.loc["Итого"]).mul(100).round(1)
cross_pct.loc["Итого"] = cross_abs.loc["Итого"]

print("\n" + "=" * 60)
print("РАСПРЕДЕЛЕНИЕ ИНН ПО КОРЗИНАМ APPROVED_SUM (% от сегмента)")
print("=" * 60)
display(cross_pct)

# --- Столбчатая диаграмма ---
plot_data = cross_abs.drop("Итого").drop(columns="Итого")
ax = plot_data.plot(kind="bar", figsize=(12, 5), edgecolor="white")
ax.set_title("Распределение ИНН по корзинам APPROVED_SUM (только RUB)", fontsize=13, fontweight="bold")
ax.set_xlabel("Корзина")
ax.set_ylabel("Количество ИНН")
ax.legend(title="Сегмент")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# --- Сводная статистика по сегментам ---
sum_stats = inn_sum.groupby("segment")["total_sum"].agg(
    ["count", "mean", "median", "min", "max"]
).reindex(seg_order)
sum_stats.columns = ["ИНН", "Среднее", "Медиана", "Мин", "Макс"]
for c in ["Среднее", "Медиана", "Мин", "Макс"]:
    sum_stats[c] = sum_stats[c].apply(lambda x: f"{x:,.0f}")

print("\n" + "=" * 60)
print("СТАТИСТИКА APPROVED_SUM ПО СЕГМЕНТАМ (RUB)")
print("=" * 60)
display(sum_stats)